# 🕵️‍♂️ Debug Lettura GAIN

Testiamo la lettura del solo parametro GAIN per isolare il problema.
- Se questo funziona, il problema era nel loop veloce (timing).
- Se questo fallisce, l'indirizzo o il comando sono errati.

In [ ]:
from boss_ir2_tool import BossIR2Manager
import time

manager = BossIR2Manager()

print("Tentativo lettura GAIN isolato...")
val = manager.read_param("GAIN", timeout=2.0)

if val is not None:
    print(f"✅ GAIN letto correttamente: {val}")
else:
    print("❌ Errore lettura GAIN (Timeout)")

## Monitor Traffico
Catturiamo cosa succede esattamente quando chiediamo il GAIN.

In [ ]:
import mido

GAIN_ADDR = [0x20, 0x00, 0x00, 0x04]

print("Cattura traffico RAW...")
with mido.open_input(manager.input_name) as inp:
    with mido.open_output(manager.output_name) as out:
        # Invia richiesta manuale
        print("📤 Invio RQ1 GAIN...")
        manager.send_rq1(GAIN_ADDR)
        
        # Leggi per 2 secondi
        start = time.time()
        while time.time() - start < 2:
            msg = inp.poll()
            if msg:
                if msg.type == 'sysex':
                    hex_msg = ' '.join(f'{b:02X}' for b in msg.data)
                    print(f"📥 RX SysEx: {hex_msg}")
                else:
                    print(f"📥 RX Other: {msg}")
            time.sleep(0.01)